In [ ]:
import torch
import os

In [ ]:
load_weights()
torch.load(os.path.join(CHECKPOINT_DIR, "AE_conv_v3.1_Wass_Reg_train_date=01-03_10-20", "ae_conv_v3.1_Wass_Reg_ep246.pth"), map_location="cpu")

In [ ]:
load_model()
model = AE_v3()
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

In [ ]:
def score_points(pts, model, criterion=None):
    import torch
    if not criterion:
        import torch
        criterion = torch.nn.MSELoss()

    out_scores = []
    for out in pts:
        out = torch.tensor(out).to(torch.float32)
        out = out.reshape(1, 1, 96)
        pred = model(out).detach()
        out = out.reshape(96)
        pred = pred.reshape(96)
        loss = criterion(out, pred).detach().numpy()
        out_scores.append(loss)
        
    return out_scores
score_points_call()

In [ ]:
plot_score()
from anomaly_detection.utils.plotting_styles import apply_global_style
apply_global_style()
plt.title("Point scorings with eval dataset")
plt.xlabel("Index")
plt.ylabel("Score")
plt.scatter(range(len(scores)), scores, label='FI Dataset')
plt.scatter(range(len(eval_score[:2])), eval_score[:2], color='black', label='Random')
plt.scatter(range(len(eval_score[2:4])), eval_score[2:4], color='red', label='Strong')
plt.scatter(range(len(eval_score[4:6])), eval_score[4:6], color='orange', label='Medium')
plt.scatter(range(len(eval_score[6:8])), eval_score[6:8], color='yellow', label='Weak')
plt.scatter(range(len(eval_score[8:])), eval_score[8:], color='darkgreen', label='Normal')
plt.axhline(y=0.4, color='orange', linestyle='-')
plt.axhline(y=0.6, color='red', linestyle='-')
plt.legend()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

scores = np.array(scores)
eval_score = np.array(eval_score)

fi_above = scores[scores > 0.5]
fi_below = scores[scores <= 0.5]

fi_above_mean = fi_above.mean() if len(fi_above) else 0
fi_below_mean = fi_below.mean() if len(fi_below) else 0

values = [fi_below_mean, fi_above_mean]
colors = ["royalblue", "royalblue"]
labels = ["FI normal (mean)", "FI outliers (mean)"]

group_colors = ["black", "red", "orange", "yellow", "darkgreen"]
group_labels = ["Random", "Strong", "Medium", "Weak", "Normal"]

for i in range(0, len(eval_score), 2):
    pair = eval_score[i:i+2]
    group_idx = i // 2
    
    for val in pair:
        values.append(val)
        colors.append(group_colors[group_idx])
        labels.append(group_labels[group_idx])

plt.figure(figsize=(12, 5))
plt.title("Point scorings with eval dataset")

x = np.arange(len(values))
plt.bar(x, values, color=colors)

plt.xticks(x, labels, rotation=45)
plt.ylabel("Score")

plt.tight_layout()
plt.show()

In [ ]:
from anomaly_detection.utils.plotting_styles import plot_comparison_w_heatmap
orig = full_dataset[244]
pred = model(orig.resize(1,1,96)).detach().squeeze().numpy()

plot_comparison_w_heatmap(orig, pred, reconstruction_heatmap=True)

In [ ]:
plot_anomalies()